<a href="https://colab.research.google.com/github/venkat-vipul/flyrank_ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/venkat-vipul/flyrank_ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Build the Feature Vector

The feature vector was constructed using performance metrics that are available before the prediction point. Numeric features were retained for modeling, while identifier columns were excluded from the learning process. Missing values were filled using the median for numeric columns to preserve as much data as possible without introducing information from future observations. This feature vector provides the model with stable, prediction-time information while avoiding known leakage sources.

In [7]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

feature_columns = [
    "clicks",
    "impressions",
    "ctr",
    "position"
]

available_features = [col for col in feature_columns if col in df.columns]

feature_vector = df[available_features].copy()

numeric_columns = feature_vector.select_dtypes(include="number").columns

feature_vector[numeric_columns] = feature_vector[numeric_columns].fillna(
    feature_vector[numeric_columns].median()
)

print("Feature Vector Shape:", feature_vector.shape)
feature_vector.head()

Feature Vector Shape: (30000, 1)


,ctr
0,0.76
1,0.05
2,0.09
3,0.49
4,0.13


## 2. Feature notes (meaning, missing, categorical, available-when?)

The feature vector contains performance metrics that are available before the prediction point. Numeric features are retained for modeling, while identifier columns are excluded. Missing numeric values are filled using the median to reduce the impact of incomplete records without introducing future information. Categorical features, when present, are encoded before model training. All selected features are available at prediction time and do not depend on future observations, ensuring they are suitable for decision-support modeling.

In [8]:
feature_summary = pd.DataFrame({
    "Feature": feature_vector.columns,
    "Data Type": feature_vector.dtypes.astype(str).values,
    "Missing Values": feature_vector.isnull().sum().values,
    "Available Before Prediction": "Yes",
    "Handling": "Median imputation (numeric)"
})

feature_summary

,Feature,Data Type,Missing Values,Available Before Prediction,Handling
0,ctr,float64,0,Yes,Median imputation (numeric)


## The Leakage Hunt

Feature leakage occurs when a model is trained using information that would not be available at prediction time. Such features can artificially improve performance during evaluation while reducing the model's reliability in production.

The dataset was reviewed for potential leakage. Features derived from future performance trends or directly related to the target variable were identified and excluded from the modeling pipeline. This ensures that the model learns only from information available before the prediction is made.

In [9]:
potential_leakage = [
    "trend_direction",
    "trend_pct"
]

existing_leakage = [col for col in potential_leakage if col in df.columns]

print("Potential Leakage Features Found:")
print(existing_leakage)

safe_features = df.drop(columns=existing_leakage, errors="ignore")

print("\nRemaining Features:", safe_features.shape[1])
safe_features.head()

Potential Leakage Features Found:
['trend_direction', 'trend_pct']

Remaining Features: 42


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5


## What I Excluded and Why

The features `trend_direction` and `trend_pct` were excluded because they contain information derived from future performance trends. Including them would introduce target leakage, resulting in overly optimistic evaluation metrics that would not generalize to unseen data.

Identifier columns such as `content_id` and `client_id` were also not considered predictive features, since they uniquely identify records rather than describing content characteristics. The final feature set includes only variables that are available before prediction and are suitable for building a reliable machine learning model.

In [10]:
excluded_features = {
    "trend_direction": "Derived from future trend information.",
    "trend_pct": "Directly related to future performance.",
    "content_id": "Identifier column.",
    "client_id": "Identifier column."
}

exclusion_summary = pd.DataFrame(
    list(excluded_features.items()),
    columns=["Excluded Feature", "Reason"]
)

exclusion_summary

,Excluded Feature,Reason
0,trend_direction,Derived from future trend information.
1,trend_pct,Directly related to future performance.
2,content_id,Identifier column.
3,client_id,Identifier column.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.